# Stock Market Social Network — Temporal Link Prediction (Improved Pipeline)

Clean rewrite of `network_pipeline_fix_1.ipynb` addressing its core issues:

| Original issue | Fix |
|---|---|
| Redundant quarterly + combined NetworkX graphs over the same data | Data lives once in a Polars frame; windows are built on demand |
| Merging quarters dropped history (duration, frequency of holdings) | Window aggregation keeps `frequency`, `duration`, `mean/std/last VALUE` per edge |
| DataFrame → nx.Graph → igraph → tensor roundtrip | Polars → torch tensors directly — one representation |
| NetworkX PageRank/HITS/closeness/Leiden don't scale | Structural signal comes from the GNN — no expensive nx centrality |
| Fund/stock identity drifted across time windows | Global `cik_id` / `cusip_id` + persistent `nn.Embedding` across all windows |
| Data and model logic tangled in one procedural pipeline | `HoldingsDataLayer` owns data access; `BipartiteGraphSAGE` owns the model |

**Sliding window:** train on N quarters → predict new fund→stock links in the next quarter. The same model object (and its embeddings) carries across windows — truly continual temporal learning, not N independent fits.


In [1]:
# Colab: install required packages (run once)
!pip install -q torch-geometric polars


In [2]:
!pip install -q sklearn torch


  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [15 lines of output]
      The 'sklearn' PyPI package is deprecated, use 'scikit-learn'
      rather than 'sklearn' for pip commands.
      
      Here is how to fix this error in the main use cases:
      - use 'pip install scikit-learn' rather than 'pip install sklearn'
      - replace 'sklearn' by 'scikit-learn' in your pip requirements files
        (requirements.txt, setup.py, setup.cfg, Pipfile, etc ...)
      - if the 'sklearn' package is used by one of your dependencies,
        it would be great if you take some time to track which package uses
        'sklearn' instead of 'scikit-learn' and report it to their issue tracker
      - as a last resort, set the environment variable
        SKLEARN_ALLOW_DEPRECATED_SKLEARN_PACKAGE_INSTALL=True to avoid this error
      
      More information is available at
      https://github.com/scikit-learn/sklearn-

In [3]:
!pip install scikit-learn


  Using cached scikit_learn-1.8.0-cp311-cp311-win_amd64.whl.metadata (11 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.8.0-cp311-cp311-win_amd64.whl (8.1 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)

   ------------- -------------------------- 1/3 [joblib]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [sci

In [4]:
import os
import warnings

import numpy as np
import pandas as pd
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from sklearn.metrics import roc_auc_score, precision_score, recall_score

warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)} | CUDA: {torch.version.cuda}")


c:\Users\potda\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


## 1. Load data

Single materialized Polars frame with **global** `cik_id` / `cusip_id` so the same fund/stock keeps the same integer id across every quarter. This is what later makes `nn.Embedding` temporally consistent.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [5]:
root = r"C:\Users\potda\Daniel\BGU\Year_D\סמסטר ז\Final_Project\Social-Network-Stock-Market\Data\holdings\holdings"

ticker_map = (
    pl.scan_parquet(f"{root}/ticker_to_cusip.parquet")
      .select(["cusip", "ticker"])
)

prices = (
    pl.scan_parquet(f"{root}/tickerprices.parquet")
      .select(["ticker", "period_start", "price"])
      .with_columns(pl.col("period_start").cast(pl.Date))
)

data = (
    pl.scan_parquet(f"{root}/holdings_filtered_part*.parquet")
      .select(["cik", "cusip", "sshprnamt", "year", "quarter", "period_start"])
      .with_columns([
          pl.col("period_start").cast(pl.Date),
          pl.when(pl.col("year").is_not_null())
            .then(pl.col("year"))
            .otherwise(pl.col("period_start").dt.year())
            .cast(pl.Int16).alias("YEAR"),
          pl.when(pl.col("quarter").is_not_null())
            .then(pl.col("quarter"))
            .otherwise(((pl.col("period_start").dt.month() - 1) // 3 + 1))
            .cast(pl.Int8).alias("QUARTER"),
          pl.col("sshprnamt").cast(pl.Int64),
      ])
      .join(ticker_map, on="cusip", how="left")
      .join(prices, on=["ticker", "period_start"], how="left")
      .drop(["year", "quarter", "ticker"])
)

# Global consistent IDs — same cik_id / cusip_id across all quarters.
# This is the key to temporal consistency: later, nn.Embedding row `k`
# is always the same fund, regardless of which window we train on.
cik_map   = data.select("cik").unique().sort("cik").with_row_count("cik_id")
cusip_map = data.select("cusip").unique().sort("cusip").with_row_count("cusip_id")

data = (
    data.join(cik_map,   on="cik",   how="left")
        .join(cusip_map, on="cusip", how="left")
        .drop(["cik", "cusip"])
        .collect()
        .filter((pl.col("YEAR") >= 2021) & (pl.col("YEAR") <= 2024))
        .with_columns([
            (pl.col("sshprnamt").cast(pl.Float64) *
             pl.col("price").cast(pl.Float64).fill_null(0.0)).alias("VALUE")
        ])
)

N_FUNDS  = int(data["cik_id"].max()) + 1
N_STOCKS = int(data["cusip_id"].max()) + 1

qs = data.select(["YEAR", "QUARTER"]).unique().sort(["YEAR", "QUARTER"]).rows()
print(f"Rows: {data.height:,}")
print(f"Funds (global): {N_FUNDS:,} | Stocks (global): {N_STOCKS:,}")
print(f"Quarters: {qs}")


SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 2-3: truncated \UXXXXXXXX escape (645292826.py, line 1)

## 2. Data Layer

`HoldingsDataLayer` owns all data access. Partitions once by (YEAR, QUARTER), then serves windows on demand.

**Key property:** when aggregating a window of quarters, each fund→stock edge carries 5 temporal features:

- `frequency`  — how many quarters within the window this holding appeared
- `duration`   — span between first and last appearance (in quarters)
- `mean_value` — average USD value across quarters
- `std_value`  — volatility of USD value
- `last_value` — most recent USD value (recency signal)

So a holding that appeared in all 8 training quarters with stable value is *not* indistinguishable from a one-time appearance — the original pipeline lost this distinction when it merged graphs.

**Bipartite packing:** fund nodes occupy `[0, N_FUNDS)`, stock nodes `[N_FUNDS, N_FUNDS + N_STOCKS)` in one unified index space. This lets a single `SAGEConv` message-pass in both directions without PyG's heterogeneous-graph overhead.


In [ ]:
class HoldingsDataLayer:
    """On-demand temporal graph builder. Polars → torch tensors in one hop."""

    def __init__(self, data: pl.DataFrame, n_funds: int, n_stocks: int, device='cpu'):
        self.n_funds = n_funds
        self.n_stocks = n_stocks
        self.n_total = n_funds + n_stocks
        self.device = device

        # Partition once; O(1) lookup per quarter thereafter.
        self._by_quarter = {}
        for part in data.partition_by(["YEAR", "QUARTER"]):
            key = (int(part["YEAR"][0]), int(part["QUARTER"][0]))
            self._by_quarter[key] = part

    def quarters(self):
        return sorted(self._by_quarter.keys())

    def _bipartite_edge_index(self, src_fund: torch.Tensor, dst_stock: torch.Tensor):
        dst = dst_stock + self.n_funds
        return torch.stack([
            torch.cat([src_fund, dst]),
            torch.cat([dst, src_fund]),
        ])

    def window_graph(self, quarters):
        """Aggregate quarters into one bipartite graph that preserves per-edge history."""
        dfs = [self._by_quarter[q] for q in quarters if q in self._by_quarter]
        if not dfs:
            return None

        combined = (
            pl.concat(dfs)
              .sort(["YEAR", "QUARTER"])
              .with_columns(
                  (pl.col("YEAR").cast(pl.Int32) * 4 + pl.col("QUARTER").cast(pl.Int32))
                  .alias("q_idx")
              )
        )

        agg = combined.group_by(["cik_id", "cusip_id"]).agg([
            pl.len().alias("frequency"),
            (pl.col("q_idx").max() - pl.col("q_idx").min() + 1).alias("duration"),
            pl.col("VALUE").mean().fill_null(0.0).alias("mean_value"),
            pl.col("VALUE").std().fill_null(0.0).alias("std_value"),
            pl.col("VALUE").last().fill_null(0.0).alias("last_value"),
        ])

        src = torch.from_numpy(agg["cik_id"].to_numpy().astype(np.int64)).to(self.device)
        dst = torch.from_numpy(agg["cusip_id"].to_numpy().astype(np.int64)).to(self.device)

        feats = np.stack([
            agg["frequency"].to_numpy().astype(np.float32),
            agg["duration"].to_numpy().astype(np.float32),
            agg["mean_value"].to_numpy().astype(np.float32),
            agg["std_value"].to_numpy().astype(np.float32),
            agg["last_value"].to_numpy().astype(np.float32),
        ], axis=1)
        # log-compress heavy-tailed monetary columns
        feats[:, 2:5] = np.sign(feats[:, 2:5]) * np.log1p(np.abs(feats[:, 2:5]))

        return {
            "edge_index": self._bipartite_edge_index(src, dst),
            "edge_attr":  torch.from_numpy(feats).to(self.device),  # [E, 5] forward-only
            "src":        src,
            "dst":        dst,
        }

    def quarter_edges(self, yq):
        """Positive fund→stock edges for one quarter (used as test targets)."""
        df = self._by_quarter.get(yq)
        if df is None or df.height == 0:
            return None
        src = torch.from_numpy(df["cik_id"].to_numpy().astype(np.int64)).to(self.device)
        dst = torch.from_numpy(df["cusip_id"].to_numpy().astype(np.int64)).to(self.device)
        return src, dst


## 3. Model Layer

`BipartiteGraphSAGE` is one object reused across *every* window. Its `nn.Embedding` tables are indexed by global `cik_id` / `cusip_id`, so:

- Fund 7's vector in window 5 is the **same row** that was updated in windows 1–4.
- Temporal drift is learned end-to-end, not reinvented per window.

No more re-keying of nodes between sliding windows.


In [ ]:
class BipartiteGraphSAGE(nn.Module):
    """Two-layer GraphSAGE over a unified bipartite node space.

    Embeddings are indexed by global (cik_id / cusip_id), so the same row persists
    across sliding windows — true temporal continuity.
    """

    def __init__(self, n_funds: int, n_stocks: int,
                 emb_dim: int = 32, hidden: int = 64, out_dim: int = 32):
        super().__init__()
        self.n_funds  = n_funds
        self.n_stocks = n_stocks
        self.n_total  = n_funds + n_stocks

        self.fund_emb  = nn.Embedding(n_funds,  emb_dim)
        self.stock_emb = nn.Embedding(n_stocks, emb_dim)
        nn.init.normal_(self.fund_emb.weight,  std=0.1)
        nn.init.normal_(self.stock_emb.weight, std=0.1)

        self.conv1 = SAGEConv(emb_dim, hidden)
        self.conv2 = SAGEConv(hidden,  out_dim)

    def node_features(self):
        return torch.cat([self.fund_emb.weight, self.stock_emb.weight], dim=0)

    def forward(self, edge_index):
        x = self.node_features()
        x = F.relu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)
        return x

    def score_pairs(self, z, fund_ids, stock_ids):
        return (z[fund_ids] * z[stock_ids + self.n_funds]).sum(dim=-1)


## 4. Training & evaluation

- **Loss weighting by edge frequency:** persistent holdings contribute a stronger gradient than one-off appearances. This is how the temporal features from `window_graph` actually influence learning without needing a fancier graph conv.
- **Test-time dedup:** edges already present in the training window are filtered out so evaluation measures prediction of *new* links — no trivial hits.
- **Continual fine-tuning:** the first window trains from scratch; subsequent windows fine-tune with lower LR. Embeddings accumulate signal across time.


In [ ]:
def sliding_windows(quarters, train_len=3, test_offset=1):
    n = len(quarters)
    for i in range(n - train_len - test_offset + 1):
        train_q = quarters[i:i + train_len]
        test_q  = quarters[i + train_len + test_offset - 1]
        yield train_q, test_q


def train_window(model, graph, device, epochs=30, lr=0.01):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    edge_index = graph["edge_index"]
    src, dst = graph["src"], graph["dst"]

    # Weight each positive by its normalized frequency in (0, 1].
    freq = graph["edge_attr"][:, 0]
    w = freq / freq.max().clamp(min=1.0)

    model.train()
    loss_val = float('nan')
    for _ in range(epochs):
        opt.zero_grad()
        z = model(edge_index)

        pos_logits = (z[src] * z[dst + model.n_funds]).sum(-1)

        n = src.size(0)
        neg_src = torch.randint(0, model.n_funds,  (n,), device=device)
        neg_dst = torch.randint(0, model.n_stocks, (n,), device=device)
        neg_logits = (z[neg_src] * z[neg_dst + model.n_funds]).sum(-1)

        pos_loss = F.binary_cross_entropy_with_logits(
            pos_logits, torch.ones_like(pos_logits), weight=w
        )
        neg_loss = F.binary_cross_entropy_with_logits(
            neg_logits, torch.zeros_like(neg_logits)
        )
        loss = pos_loss + neg_loss
        loss.backward()
        opt.step()
        loss_val = loss.item()

    model.eval()
    with torch.no_grad():
        z = model(edge_index)
    return z, loss_val


def evaluate_window(model, z, data_layer, train_graph, test_q, device):
    """AUC / precision / recall on NEW links in the next quarter."""
    test = data_layer.quarter_edges(test_q)
    if test is None:
        return None
    src_t, dst_t = test

    # Vectorized "test edge not in training window" via packed pair ids.
    pack = data_layer.n_stocks
    train_pack = (train_graph["src"] * pack + train_graph["dst"]).to(device)
    test_pack  = (src_t * pack + dst_t).to(device)
    mask = ~torch.isin(test_pack, train_pack)

    src_t, dst_t = src_t[mask], dst_t[mask]
    if src_t.numel() == 0:
        return None

    with torch.no_grad():
        pos = torch.sigmoid(model.score_pairs(z, src_t, dst_t))

        n = src_t.size(0)
        neg_src = torch.randint(0, model.n_funds,  (n,), device=device)
        neg_dst = torch.randint(0, model.n_stocks, (n,), device=device)
        neg = torch.sigmoid(model.score_pairs(z, neg_src, neg_dst))

    y_true = torch.cat([torch.ones(n, device=device),
                        torch.zeros(n, device=device)]).cpu().numpy()
    y_pred = torch.cat([pos, neg]).cpu().numpy()
    y_hat  = (y_pred > 0.5).astype(int)

    return {
        "auc":         float(roc_auc_score(y_true, y_pred)),
        "precision":   float(precision_score(y_true, y_hat, zero_division=0)),
        "recall":      float(recall_score(y_true, y_hat, zero_division=0)),
        "n_new_links": int(n),
    }


## 5. Run sliding-window pipeline

One `HoldingsDataLayer`, one `BipartiteGraphSAGE`. Windows are built on demand; the model keeps learning across them.


In [ ]:
data_layer = HoldingsDataLayer(data, N_FUNDS, N_STOCKS, device=device)
model      = BipartiteGraphSAGE(N_FUNDS, N_STOCKS).to(device)

TRAIN_WINDOW = 3
TEST_OFFSET  = 1
MODELS_DIR   = "temporal_models"
os.makedirs(MODELS_DIR, exist_ok=True)

quarters = data_layer.quarters()
print(f"Quarters available: {quarters}\n")

results = []
for idx, (train_q, test_q) in enumerate(sliding_windows(quarters, TRAIN_WINDOW, TEST_OFFSET)):
    print(f"── Window {idx+1} | train={train_q} | test={test_q} ──")

    graph = data_layer.window_graph(train_q)
    if graph is None:
        print("  skipped (empty window)\n")
        continue

    epochs = 30  if idx == 0 else 10
    lr     = 0.01 if idx == 0 else 0.001
    z, loss = train_window(model, graph, device, epochs=epochs, lr=lr)

    metrics = evaluate_window(model, z, data_layer, graph, test_q, device)
    if metrics is None:
        print(f"  loss={loss:.4f} — skipped (no new test links)\n")
        continue

    print(f"  loss={loss:.4f} | AUC={metrics['auc']:.4f} "
          f"| P={metrics['precision']:.4f} | R={metrics['recall']:.4f} "
          f"| new_links={metrics['n_new_links']:,}\n")

    torch.save({
        "model_state": model.state_dict(),
        "train_q":     train_q,
        "test_q":      test_q,
        "metrics":     metrics,
    }, os.path.join(MODELS_DIR, f"window_{idx+1}_{test_q[0]}Q{test_q[1]}.pt"))

    results.append({
        "window":  idx + 1,
        "year":    test_q[0],
        "quarter": test_q[1],
        **metrics,
    })

results_df = pd.DataFrame(results)
print("── Summary ──")
print(results_df)


In [ ]:
if not results_df.empty:
    print(f"Windows evaluated : {len(results_df)}")
    print(f"Mean AUC          : {results_df['auc'].mean():.4f}")
    print(f"Mean Precision    : {results_df['precision'].mean():.4f}")
    print(f"Mean Recall       : {results_df['recall'].mean():.4f}")
